# ch03 RNN 手写（练习版）

> 补全所有 `____`，让每个代码块末尾的 `assert` 全绿。
> 数据：`data/load_curve.csv` 的 A 台区（居民型），2025 全年 8760 个整点。
>
> **每题都跟了 `# 提示：`**，照着提示能独立做完。
> 如果 assert 的数字对不上，**先回头看上一步的输出**，别急着改 assert —— 那些数字都是实跑出来的。
>
> 建议顺序：先把 3.1 的滑窗搭对，再进 3.2 / 3.3 的手写对账。
> 手写对账这两个任务，**对不上就是有 bug**，别用 `abs() < 1` 糊过去。

## 任务清单

| 任务 | 主题 | 挖空 | 对应讲解小节 |
|---|---|---|---|
| 1 | 滑窗 → 监督矩阵（标签是窗口后一格） | 4 | 3.1 / 3.1.1 |
| 2 | 按时间切分 + 只用训练段标准化 | 7 | 3.1 |
| 3 | 手写 RNN 单步前向，与 `nn.RNN` 对账 | 10 | 3.2 |
| 4 | `batch_first` 与 `h_n` 的形状 | 5 | 3.2.1 |
| 5 | 手写 BPTT，与 `torch.autograd` 对账 | 12 | 3.3 |
| 6 | BPTT 的两个经典写法错误 | 7 | 3.3.1 |
| 7 | `nn.RNN` 回归器训练 + 基线对照 | 11 | 3.4 |
| 8 | 预测曲线 + 损失曲线 | 2 | 3.4 |
| 9 | 梯度消失：长度扫描 | 3 | 3.5 |
| 10 | 梯度消失：谱半径扫描 | 3 | 3.5 |
| 11 | 实测每步因子与 tanh 饱和 | 4 | 3.5.1 |
| 12 | 谱半径 1.2 也救不了 | 3 | 3.5.2 |
| **合计** | 12 个任务 / 32 个 `@@todo` 块 | **71** | |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

DATA = Path("data")          # notebook 的 cwd = coding/05_timeseries
CSV = DATA / "load_curve.csv"

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

raw = pd.read_csv(CSV, encoding="utf-8-sig", parse_dates=["时间戳"])
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "| MPS", torch.backends.mps.is_available(), "（本章全程 CPU，数据小）")
print("原始行数", len(raw), "| 列", list(raw.columns))

In [ ]:
df = raw.drop_duplicates().sort_values("时间戳").reset_index(drop=True)
print("去重排序后", len(df), "行")

# 本章只做主序列：A 台区（居民型，20 点单峰、周末更忙），2025 全年 8760 个整点
a_frame = df[df["台区编号"] == "STATION_A_01"].set_index("时间戳").sort_index()
s_a = a_frame["负荷值"].asfreq("h").loc["2025-01-01":"2025-12-31 23:00"]
print("A 台区整点 %d | 缺失 %d" % (len(s_a), int(s_a.isna().sum())))
print(s_a.head(3))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

# macOS 上 matplotlib 默认字体不含中文，不设置的话图上全是方块
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "STHeiti"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib", matplotlib.__version__, "| 中文字体 PingFang SC")

In [ ]:
# 脚手架：下面的常量与工具函数两版都有，你只填 @@todo 块里的内容
W = 24            # 滑窗长度：用过去 24 小时预测下 1 小时
HID = 12          # 隐藏单元数
EPOCHS = 15
BATCH = 128
SEED = 42


def clean_series(s):
    """按**小时分组**的中位数做 3 倍阈值判异常，再线性插值。"""
    med = s.groupby(s.index.hour).transform("median")
    return s.where(s.between(0, 3 * med)).interpolate()


def make_windows(z, w):
    """一维序列 → 监督矩阵：X[i] = z[i : i+w]，y[i] = z[i+w]。

    **标签是窗口后面那一格**，不是窗口末格 —— 这是时序监督化的第一号坑。
    """
    x = np.lib.stride_tricks.sliding_window_view(z, w)[:-1]
    return x, z[w:]


def make_rnn(rho, hidden=16, seed=1):
    """造一个 W_h 谱半径被钉死在 rho 上的 torch RNN（3.5 的梯度实验专用）。"""
    with torch.random.fork_rng():
        torch.manual_seed(seed)
        m = nn.RNN(1, hidden, batch_first=True).double()
    with torch.no_grad():
        w = m.weight_hh_l0
        m.weight_hh_l0.copy_(w * (rho / float(torch.linalg.eigvals(w).abs().max())))
    return m


def last_step_grads(model, length, seed=3, scale=1.0):
    """loss 只挂在**最后一步**的输出上，回传得到每个输入步的梯度幅度。

    为什么要"只挂最后一步"：若 loss 挂在所有步上，dL/dx_t 里会有
    dL/dh_t 这一项**本步**的贡献，它一个时间步都不用传，永远不衰减 --
    测出来的比值会是 0.3 这种"假的不衰减"。序列到一维的回归任务天然只有
    最后一步有 loss，本函数就是复现这个口径。
    """
    with torch.random.fork_rng():
        torch.manual_seed(seed)
        x = torch.randn(1, length, 1, dtype=torch.float64, requires_grad=True) * scale
    out, _ = model(x)
    g = torch.autograd.grad(out[:, -1, :].pow(2).sum(), x)[0]
    return g[0, :, 0].abs().detach().numpy()

## 任务 1：滑窗 → 监督矩阵（3.1）

In [ ]:
# TODO(1)：清洗 A 台区序列（脚手架已有 clean_series）
s_a_clean = ____

SERIES = s_a_clean.values.astype("float64")
print("主序列 %d 点 | 均值 %.6f | 标准差 %.6f | min %.4f | max %.4f"
      % (len(SERIES), SERIES.mean(), SERIES.std(), SERIES.min(), SERIES.max()))

# TODO(2)：把一维序列切成监督矩阵 X / y（脚手架已有 make_windows）
# 提示：先把形状与错位关系搭对，标准化留到下一步再做
X_all, y_all = ____

# TODO(3)：手工验证「标签 = 窗口后一格」这条关系
# 提示：窗口里的最后一格是 X_all[0, -1]，标签 y_all[0] 应该是它的**下一格**
last_in_win = ____
first_label = ____

print("X %s | y %s | 样本数 %d" % (X_all.shape, y_all.shape, len(y_all)))
print("X[0] 前 3 格 %s ... 窗口末格 %.4f" % (np.round(X_all[0, :3], 3), last_in_win))
print("标签 y[0] %.4f = SERIES[%d] %.4f（窗口末格是 SERIES[%d] = %.4f）"
      % (first_label, W, SERIES[W], W - 1, SERIES[W - 1]))

# ---- 验收 ----
assert X_all.shape == (8736, 24) and y_all.shape == (8736,), "8760 - 24 = 8736 个样本"
assert np.array_equal(X_all[0], SERIES[:W])
assert last_in_win == SERIES[W - 1] and first_label == SERIES[W]
assert abs(first_label - last_in_win) > 1e-9, "标签与窗口末格是两个不同的时刻"

## 任务 2：时间切分与训练段标准化（3.1）

In [ ]:
# TODO(4)：先在**序列**级别按时间 8:2 切分
# 提示：时序数据只有一种合法切法：按时间，绝不能随机抽
cut_pt = ____

# TODO(5)：标准化参数**只允许来自训练段**
# 提示：训练段就是 SERIES[:cut_pt]；用全量均值/方差就是 ch01 说的第三种泄漏
MU = ____
SD = ____

Z = (SERIES - MU) / SD

# TODO(6)：在标准化后的序列上重新切窗，然后按样本切分
# 提示：样本级切分点 = 序列级切分点 - 窗口长度（窗口要满 24 格才有一个标签）
Xz, yz = ____
n_train = ____
Xtr, ytr = ____
Xte, yte = ____

print("序列级切分点 %d（%s）| 样本级切分点 %d" % (cut_pt, s_a_clean.index[cut_pt], n_train))
print("训练 %s / 测试 %s" % (Xtr.shape, Xte.shape))
print("y 训练段 mean %.6f std %.6f | 测试段 mean %.6f std %.6f"
      % (ytr.mean(), ytr.std(), yte.mean(), yte.std()))

# ---- 验收 ----
assert len(SERIES) == 8760 and Xz.shape == (8736, 24)
assert n_train == 6984 and Xte.shape == (1752, 24)
assert abs(MU - 454.561801) < 1e-6 and abs(SD - 72.623128) < 1e-6
assert abs(float(ytr.mean()) - 0.001420) < 1e-6, "训练段标准化后均值几乎为 0"
assert abs(float(ytr.std()) - 1.000632) < 1e-6
assert abs(float(yte.mean()) - 0.388310) < 1e-6, "测试段是 10/20 以后的冬季，均值被抬到 +0.39"

## 任务 3：手写 RNN 单步前向并对账（3.2）

In [ ]:
# 用一组固定的小输入做对账：B=3 个样本、T=7 个时间步、D=2 维特征、H=5 个隐藏单元
T_DEMO, B_DEMO, D_DEMO, H_DEMO = 7, 3, 2, 5

with torch.random.fork_rng():
    torch.manual_seed(0)          # 随机性一律显式固定，且包进 fork_rng

    # TODO(7)：造固定输入，并建一个**双精度**的 nn.RNN 当参照
    # 提示：torch.randn(B, T, D, dtype=torch.float64) 造输入
    # 提示：.double() 是为了让逐值对账落到 1e-12 以下（float32 只有 1e-7 精度）
    x_demo = ____
    ref = ____

out_ref, hn_ref = ref(x_demo)
print("nn.RNN 的 out %s | h_n %s（h_n 第 0 维 = 层数 x 方向数）"
      % (tuple(out_ref.shape), tuple(hn_ref.shape)))

# TODO(8)：把 torch 的权重与偏置搬进 numpy
# 提示：weight_ih_l0 形状 (H, D)，weight_hh_l0 形状 (H, H)
# 提示：**偏置有两个**（bias_ih_l0 与 bias_hh_l0），公式里的 b_h 是它们之和
Wx = ____
Wh = ____
bh = ____
xn = ____

# TODO(9)：初始化隐状态，并准备收集每一步的 h
# 提示：h 的形状是 (批大小, 隐藏维) —— 注意不是 (隐藏维,)
# 提示：每一步的 h 都要收集，最后 stack 成 (B, T, H) 才能和 out_ref 比
h_np = ____
hs_np = ____

for t in range(T_DEMO):
    # TODO(10)：手写单步 RNN：h_t = tanh(x_t . W_x + h_{t-1} . W_h + b_h)
    # 提示：x_t 就是 xn[:, t, :]，形状 (B, D)；矩阵乘用 @
    # 提示：torch 的权重是 (out, in) 排布，所以转置用 .T
    h_np = ____
    ____

# torch 的 out 是 (B, T, H)，stack 出来是 (T, B, H)，所以要转置
hs_np = np.stack(hs_np).transpose(1, 0, 2)

d_fwd = float(np.abs(hs_np - out_ref.detach().numpy()).max())
d_hn = float(np.abs(hs_np[:, -1] - hn_ref.detach().numpy()[0]).max())
print("手写前向 vs nn.RNN：全部时间步 max|diff| = %.3e" % d_fwd)
print("手写最后一步 h_T vs nn.RNN 的 h_n：max|diff| = %.3e" % d_hn)
print("权重形状 W_x %s | W_h %s | b_h %s" % (Wx.shape, Wh.shape, bh.shape))

# ---- 验收 ----
assert hs_np.shape == (B_DEMO, T_DEMO, H_DEMO)
assert d_fwd < 1e-12, "双精度下手写前向必须与 nn.RNN 逐值一致（实测 ~1e-16）"
assert d_hn < 1e-12, "h_n 就是最后一个时间步的 h —— 不是另算出来的东西"

## 任务 4：`batch_first` 与 `h_n` 的形状（3.2.1）

In [ ]:
with torch.random.fork_rng():
    torch.manual_seed(0)

    # TODO(11)：关掉 batch_first，看输出形状与输入方向怎么变
    # 提示：batch_first=False 时输入被解释成 (T, B, D)，所以要 transpose(0, 1)
    # 提示：权重一模一样（load_state_dict 把 ref 的参数拷过来），结果应当只差一个转置
    ref_tf = ____
    ____
    out_tf, hn_tf = ____

d_tf = float(np.abs(out_tf.transpose(0, 1).detach().numpy() - out_ref.detach().numpy()).max())
print("batch_first=False 的 out 形状 %s（batch_first=True 是 %s）"
      % (tuple(out_tf.shape), tuple(out_ref.shape)))
print("两者转置后 max|diff| = %.3e" % d_tf)

# TODO(12)：记下两件事：h_n 的第 0 维是什么，W_x 的形状是什么
# 提示：h_n 形状是 (num_layers * num_directions, B, H)
hn_shape_1layer = ____
Wx_rows = ____

print("h_n 形状 %s（第 0 维 = 层数 x 方向数）| W_x 形状 %s = (H, D)" % (hn_shape_1layer, Wx_rows))

# ---- 验收 ----
assert tuple(out_tf.shape) == (T_DEMO, B_DEMO, H_DEMO)
assert d_tf < 1e-12, "batch_first 只改输入/输出的排布，不改数值"
assert hn_shape_1layer == (1, B_DEMO, H_DEMO), "单层单向时 h_n 第 0 维是 1"
assert Wx_rows == (H_DEMO, D_DEMO) and Wh.shape == (H_DEMO, H_DEMO)

## 任务 5：手写 BPTT 并对账（3.3）

In [ ]:
# TODO(13)：先用 torch 的 autograd 算一遍「loss = 所有输出平方和」的梯度
# 提示：clone().requires_grad_(True)：留一份可求导的输入副本
# 提示：求导前先 ref.zero_grad()，否则 .grad 会跨轮累加
x_grad = ____
out_g, _ = ____
loss_ref = ____
____
____

# TODO(14)：numpy 侧算同一个 loss（必须完全一致，否则后面比的不是同一条曲线）
loss_np = ____

# 反向要按时间**倒序**，所以递推变量初始化在循环外
d_out = 2 * hs_np                                                # d loss / d h_t = 2 h_t
h_prev = np.concatenate([np.zeros((B_DEMO, 1, H_DEMO)), hs_np[:, :-1, :]], axis=1)
dh_next = np.zeros((B_DEMO, H_DEMO))
dWx = np.zeros_like(Wx)
dWh = np.zeros_like(Wh)
db = np.zeros_like(bh)

for t in reversed(range(T_DEMO)):
    # TODO(15)：第 t 步总梯度 =「本步输出的贡献」+「来自下一步的 dh」
    # 提示：漏掉 d_out[:, t, :] 这一项，梯度会直接退化成全 0（见 3.3.1）
    dh = ____
    # TODO(16)：穿过 tanh：tanh'(a) = 1 - tanh(a)^2 = 1 - h_t^2
    da = ____
    # TODO(17)：累计三个参数的梯度：x_t 与 h_{t-1} 各自乘在自己那一侧
    # 提示：da 形状 (B, H)，x_t 与 h_{t-1} 都是 (B, .)，所以写 da.T @ ...
    dWx = ____
    dWh = ____
    db = ____
    # TODO(18)：把梯度递给上一步：dh_{t-1} = da_t . W_h
    # 提示：这里**不要**再乘一次 tanh' —— tanh' 是 a_{t-1} 那一层的事（见 3.3.1）
    dh_next = ____

dWx_ref = ref.weight_ih_l0.grad.numpy()
dWh_ref = ref.weight_hh_l0.grad.numpy()
db_ref = ref.bias_ih_l0.grad.numpy()

print("loss：手写 %.10f vs torch %.10f" % (loss_np, float(loss_ref.detach())))
print("dW_x max|diff| %.3e" % np.abs(dWx - dWx_ref).max())
print("dW_h max|diff| %.3e" % np.abs(dWh - dWh_ref).max())
print("db   max|diff| %.3e（对照的是 bias_ih_l0.grad）" % np.abs(db - db_ref).max())
print("梯度整体量级：|dW_x| %.4f | |dW_h| %.4f" % (np.linalg.norm(dWx), np.linalg.norm(dWh)))

# ---- 验收 ----
assert abs(loss_np - float(loss_ref.detach())) < 1e-9
assert np.abs(dWx - dWx_ref).max() < 1e-12, "手写 BPTT 的 dW_x 必须与 autograd 逐值一致"
assert np.abs(dWh - dWh_ref).max() < 1e-12, "dW_h 同理"
assert np.abs(db - db_ref).max() < 1e-12
assert float(np.abs(ref.bias_ih_l0.grad.numpy() - ref.bias_hh_l0.grad.numpy()).max()) < 1e-12, \
    "b_ih 与 b_hh 在公式里是相加的，所以两者的梯度**完全相同**"

## 任务 6：BPTT 的两个经典写法错误（3.3.1）

In [ ]:
# ---- 错误示范 A：dh 只取 dh_next，漏掉「本步输出」的贡献 ----
dh_next_a = np.zeros((B_DEMO, H_DEMO))
dWx_a = np.zeros_like(Wx)
dWh_a = np.zeros_like(Wh)
for t in reversed(range(T_DEMO)):
    # TODO(19)：只保留 dh_next（这是错的），穿过 tanh 后累计梯度
    da = ____
    dWx_a = ____
    dWh_a = ____
    dh_next_a = da @ Wh

# ---- 错误示范 B：把 tanh' 多乘了一次到 dh_{t-1} 上 ----
dh_next_b = np.zeros((B_DEMO, H_DEMO))
dWx_b = np.zeros_like(Wx)
dWh_b = np.zeros_like(Wh)
for t in reversed(range(T_DEMO)):
    # TODO(20)：本步处理是对的，但递推写成「da . W_h 之后再乘一次 (1 - h_t^2)」
    # 提示：这格就是要复现这个 bug，别"顺手改对"
    dh = ____
    da = ____
    dWx_b = ____
    dWh_b = ____
    dh_next_b = (da @ Wh) * (1 - hs_np[:, t, :] ** 2)

rel_a = float(np.linalg.norm(dWx_a - dWx) / np.linalg.norm(dWx))
rel_b = float(np.linalg.norm(dWx_b - dWx) / np.linalg.norm(dWx))
rel_bh = float(np.linalg.norm(dWh_b - dWh) / np.linalg.norm(dWh))
print("A 漏掉 dh_out：|dW_x| = %.6e（正确 %.6f）相对误差 %.4f"
      % (np.linalg.norm(dWx_a), np.linalg.norm(dWx), rel_a))
print("B 多乘一次 tanh'：相对误差 dW_x %.4f | dW_h %.4f" % (rel_b, rel_bh))
print("B 的 dW_h 范数 %.6f vs 正确 %.6f（只差 %.2f%%，却已经错了）"
      % (np.linalg.norm(dWh_b), np.linalg.norm(dWh),
         100 * abs(np.linalg.norm(dWh_b) / np.linalg.norm(dWh) - 1)))

# ---- 验收 ----
assert np.linalg.norm(dWx_a) < 1e-30, "漏掉本步 dh_out 后，除最后一步外没有任何梯度来源"
assert rel_a > 0.999
assert 0.05 < rel_b < 0.20, "多乘一次 tanh' 只让梯度偏了约 9% —— 这种错不会报错，只会悄悄变差"

## 任务 7：`nn.RNN` 回归器训练 + 基线对照（3.4）

In [ ]:
class RNNRegressor(nn.Module):
    """序列到一维：只取最后一步的隐状态，接一个线性头。"""

    def __init__(self, hidden=HID):
        super().__init__()
        self.rnn = nn.RNN(1, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])


# TODO(21)：把 numpy 监督矩阵转成 torch 张量，特征维补成 1
# 提示：RNN 要的是 (B, T, D)；Xtr 是 (N, T)，所以再 unsqueeze(-1)
# 提示：训练用 float32 就够了，不必 double
Xtr_t = ____
ytr_t = ____
Xte_t = ____

loss_hist = []
with torch.random.fork_rng():
    torch.manual_seed(SEED)
    model = RNNRegressor()
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    lossf = nn.MSELoss()
    for epoch in range(EPOCHS):
        perm = torch.randperm(len(Xtr_t))
        for i in range(0, len(Xtr_t), BATCH):
            idx = perm[i: i + BATCH]
            # TODO(22)：一个 batch 的四行：清零 → 前向算 loss → 反向 → 更新
            ____
            loss = ____
            ____
            ____
        loss_hist.append(float(loss.detach()))

model.eval()
with torch.no_grad():
    # TODO(23)：在测试段上前向，并把输出压平成一维
    # 提示：model(Xte_t) 形状 (N, T, 1)，用 .numpy().ravel() 变成 (N,)
    pred_te = ____

# TODO(24)：算三种 MAE —— 你的模型、朴素持续、训练段均值
# 提示：朴素基线 = 直接拿窗口最后一格当预测（Xte[:, -1]）
# 提示：另一个基线 = 拿训练段均值当预测（ytr.mean()）
mae = ____
mae_persist = ____
mae_mean = ____

print("训练 loss（每 epoch 最后一个 batch）%.6f → %.6f" % (loss_hist[0], loss_hist[-1]))
print("测试 MAE（标准化空间）%.6f | 换算回 kW %.6f" % (mae, mae * SD))
print("对照：朴素持续 %.6f | 训练段均值 %.6f" % (mae_persist, mae_mean))
print("参数个数 %d（24 个时间步共用同一份权重，所以参数量与序列长度无关）"
      % sum(p.numel() for p in model.parameters()))

# ---- 验收 ----
assert abs(mae - 0.259707) < 1e-4, "固定 seed 后结果可复现"
assert abs(mae_persist - 0.414592) < 1e-6 and abs(mae_mean - 0.733844) < 1e-6
assert mae < mae_persist, "RNN 必须打赢「上一小时」这个朴素基线"

## 任务 8：预测曲线与损失曲线（3.4）

In [ ]:
N_SHOW = 168          # 一周 = 168 小时

# TODO(25)：取测试段开头一周，把标准化空间还原成 kW
# 提示：反标准化：x * SD + MU
pred_kw = ____
true_kw = ____

fig, axes = plt.subplots(2, 1, figsize=(12, 6.5))
axes[0].plot(np.arange(N_SHOW), true_kw, lw=1.8, label="真值")
axes[0].plot(np.arange(N_SHOW), pred_kw, lw=1.4, label="RNN 预测（测试段第 1 周）")
axes[0].set_ylabel("负荷 kW")
axes[0].set_xlabel("测试段小时序号")
axes[0].set_title("RNN 单步预测 vs 真值（测试段 MAE %.2f kW）" % (mae * SD))
axes[0].legend(loc="upper right")
axes[1].plot(np.arange(1, EPOCHS + 1), loss_hist, marker="o", color="crimson")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("MSE（标准化空间）")
axes[1].set_title("训练损失下降曲线")
plt.tight_layout()

# ---- 验收 ----
assert len(pred_kw) == N_SHOW == 168
assert abs(float(true_kw.mean()) - 438.223571) < 1e-3, \
    "测试段第 1 周（10/20 起）均值 438.22 kW，比全年均值 460.20 略低"
assert loss_hist[-1] < loss_hist[0], "15 个 epoch 内损失必须下降"

## 任务 9：梯度消失 —— 长度扫描（3.5）

In [ ]:
RNN_LENS = [10, 20, 40, 80, 160]

# TODO(26)：造一个 W_h 谱半径被钉死在 0.9 的参照 RNN（不训练，只看梯度）
# 提示：脚手架 make_rnn(rho)
rnn09 = ____

print("谱半径实测 %.6f" % float(torch.linalg.eigvals(rnn09.weight_hh_l0.detach()).abs().max()))

ratio_len = {}
for T in RNN_LENS:
    # TODO(27)：取该长度下逐步的梯度幅度，算出「第 1 步 / 第 T 步」的比值
    # 提示：脚手架 last_step_grads(model, length) 返回长度 T 的逐输入步梯度幅度
    g = ____
    ratio_len[T] = ____

for T in RNN_LENS:
    r = ratio_len[T]
    print("T=%3d  第1步/第T步 = %.3e   |  平均每步 x%.4f" % (T, r, r ** (1 / (T - 1))))

# ---- 验收 ----
assert all(ratio_len[a] > ratio_len[b] for a, b in zip(RNN_LENS[:-1], RNN_LENS[1:])), \
    "序列越长，早期梯度越小"
assert ratio_len[10] > 0.05 and ratio_len[160] < 1e-11
assert abs(np.log10(ratio_len[160]) - (-12.017)) < 0.5, "T=160 时早期梯度已经掉了 12 个数量级"

## 任务 10：梯度消失 —— 谱半径扫描（3.5）

In [ ]:
RHOS = [0.5, 0.7, 0.9, 0.99, 1.2]

ratio_rho = {}
for rho in RHOS:
    # TODO(28)：造谱半径为 rho 的 RNN，在 T=160 上量「第 1 步 / 第 T 步」梯度比
    m = ____
    g = ____
    ratio_rho[rho] = ____

for rho in RHOS:
    r = ratio_rho[rho]
    print("rho=%.2f  第1步/第T步 = %.3e   平均每步 x%.4f" % (rho, r, r ** (1 / 159)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].semilogy(RNN_LENS, [ratio_len[T] for T in RNN_LENS], marker="o", color="tab:blue")
axes[0].set_xlabel("序列长度 T")
axes[0].set_ylabel("第 1 步 / 第 T 步 梯度比（对数轴）")
axes[0].set_title("梯度消失：长度每翻一倍，早期梯度掉几个量级")
axes[0].grid(alpha=0.3)
axes[1].semilogy(RHOS, [ratio_rho[r] for r in RHOS], marker="s", color="tab:red")
axes[1].set_xlabel("W_h 谱半径 rho")
axes[1].set_ylabel("第 1 步 / 第 T 步 梯度比（对数轴）")
axes[1].set_title("衰减率由 W_h 的谱半径决定（T=160）")
axes[1].grid(alpha=0.3)
plt.tight_layout()

# ---- 验收 ----
assert all(ratio_rho[a] < ratio_rho[b] for a, b in zip(RHOS[:-1], RHOS[1:])), "谱半径越大越不容易衰减"
assert ratio_rho[0.5] < 1e-40 and ratio_rho[0.99] > 1e-8
assert all((ratio_rho[r] ** (1 / 159)) < 1.0 for r in RHOS), \
    "连 rho=1.2 也是衰减的 —— tanh 把它按住了，见 3.5.2"

## 任务 11：实测每步因子与 tanh 饱和（3.5.1）

In [ ]:
# TODO(29)：实测「平均每步衰减因子」，再和理论上的谱半径对照（T=40）
# 提示：因子 = 首末比的 1/(T-1) 次方
meas_factor = ____
theory = ____

print("rho=0.9 时：理论每步因子 = %.4f | 实测每步因子 = %.4f | 差 %.4f"
      % (theory, meas_factor, theory - meas_factor))
print("差的这部分就是 tanh' —— 链式法则里每穿过一次 tanh 就要乘一个 (1 - h^2) <= 1")

# rho=0.9，只把**输入幅度**放大，看衰减变快还是变慢（T=40）
ratio_scale = {}
for sc in (1.0, 5.0, 20.0):
    # TODO(30)：同口径量一次，只是把输入幅度乘上 sc
    # 提示：脚手架 last_step_grads(model, length, scale=...)
    g_s = ____
    ratio_scale[sc] = ____
for sc in (1.0, 5.0, 20.0):
    r = ratio_scale[sc]
    print("输入幅度 x%-5.1f  首末比 %.3e  平均每步 x%.4f" % (sc, r, r ** (1 / 39)))

# ---- 验收 ----
assert abs(meas_factor - 0.753637) < 1e-4, "实测因子比谱半径 0.9 小 0.146，就是 tanh' 的代价"
assert ratio_scale[1.0] > ratio_scale[5.0] > ratio_scale[20.0], \
    "输入越大 → h 越饱和 → tanh' 越小 → 衰减越快"
assert ratio_scale[20.0] < 1e-13

## 任务 12：谱半径 1.2 也救不了（3.5.2）

In [ ]:
# TODO(31)：把 rho 调到 1.2（谱半径 > 1，理论上该"放大"了）
# 提示：只要改一个数字：make_rnn(1.2)
rnn12 = ____

# 与上一格完全同口径，只换了 rho
ratio_scale12 = {}
for sc in (1.0, 5.0, 20.0):
    # TODO(32)：在 rho=1.2 的模型上再量一遍三个输入幅度
    g_s = ____
    ratio_scale12[sc] = ____

for sc in (1.0, 5.0, 20.0):
    r = ratio_scale12[sc]
    print("rho=1.2 输入幅度 x%-5.1f  首末比 %.3e  平均每步 x%.4f" % (sc, r, r ** (1 / 39)))

print()
print("结论对照：")
print("  rho=0.9  x1 的每步因子 %.4f  →  x20 变成 %.4f"
      % (ratio_scale[1.0] ** (1 / 39), ratio_scale[20.0] ** (1 / 39)))
print("  rho=1.2  x1 的每步因子 %.4f  →  x20 变成 %.4f"
      % (ratio_scale12[1.0] ** (1 / 39), ratio_scale12[20.0] ** (1 / 39)))

# ---- 验收 ----
assert ratio_scale12[1.0] > ratio_scale[1.0], "谱半径越大，同样输入下越不容易衰减"
assert ratio_scale12[20.0] < 1e-8, "但只要输入一大，rho=1.2 也救不回来 —— 饱和"
assert (ratio_scale12[20.0] ** (1 / 39)) < 0.60
assert ratio_scale12[1.0] < 1.0, "谱半径 1.2 也没能让梯度放大（tanh 饱和封顶）"

## 自查清单（能不看笔记答出来才算过）

- [ ] 12 个代码块的 assert 全部通过
- [ ] 8736 这个样本数怎么来的？`X` 的形状为什么是 `(8736, 24)`？
- [ ] 标签为什么必须取「窗口后一格」？写成窗口末格会发生什么？
- [ ] 公式里的 `b_h` 对应 torch 的哪两个属性？
- [ ] torch 的 `weight_ih_l0` 是 `(H, D)` 还是 `(D, H)`？手写时要怎么处理？
- [ ] `batch_first=True` 与 `False` 的输出差在哪？数值上差多少？
- [ ] `h_n` 的形状是什么？单层单向时为什么不是 `(B, H)`？
- [ ] BPTT 里 `dh_t` 由哪两部分组成？漏掉其中一部分会怎样？
- [ ] `dh_{t-1} = da_t · W_h` 后面还要不要乘 `tanh'`？多乘会偏多少？
- [ ] 为什么 `bias_ih_l0` 与 `bias_hh_l0` 的梯度完全一样？
- [ ] 为什么梯度实验的 loss 必须只挂最后一步？
- [ ] T=160、ρ=0.9 时，第 1 步与第 T 步的梯度比是多少？
- [ ] 实测每步因子 0.7536 与谱半径 0.9 的差从哪来？
- [ ] 为什么把 `W_h` 谱半径调到 1.2 也解决不了梯度消失？

## 关键真值对照（做完再对）

| 量 | 值 |
|---|---|
| 主序列长度 / 样本数 | 8760 / 8736 |
| `X` 形状 | (8736, 24) |
| 训练段 MU / SD | 454.561801 / 72.623128 |
| 样本级切分点 / 测试样本数 | 6984 / 1752 |
| y 训练段 mean / 测试段 mean | 0.001420 / 0.388310 |
| 手写前向 vs `nn.RNN` max&#124;diff&#124; | 1.388e-16 |
| 手写 BPTT vs autograd max&#124;diff&#124; | dW_x 1.776e-15 / dW_h 8.882e-16 |
| 错误示范 A（漏 dh_out）相对误差 | 1.0000（梯度全 0） |
| 错误示范 B（多乘 tanh'）相对误差 | dW_x 0.0891 / dW_h 0.1326 |
| RNN 测试 MAE（标准化空间 / kW） | 0.259707 / 18.86 |
| 朴素持续 MAE / 训练段均值 MAE | 0.414592 / 0.733844 |
| T=160、ρ=0.9 首末梯度比 | 9.615e-13 |
| ρ 扫描（T=160）：0.5 / 0.7 / 0.9 / 0.99 / 1.2 | 2.46e-52 / 2.45e-29 / 9.62e-13 / 4.41e-07 / 1.08e-03 |
| ρ=0.9 实测每步因子（T=40） | 0.753637 |
| ρ=0.9 输入 ×1 / ×5 / ×20 每步因子 | 0.7536 / 0.6041 / 0.4392 |
| ρ=1.2 输入 ×1 / ×5 / ×20 每步因子 | 0.9913 / 0.7948 / 0.5698 |